<center><br><font size=10>HW2</font><br>
<font size=6>Linear regression and the Bias-Variance Tradeoff</font>
<br><br>
<b>Introduction to Machine Learning – Digital Sciences for High-Tech</b>
<br><i>Spring 2025</i></center>

# Instructions
1. Write all code within the functions
2. Make sure to **return** the solution in every function
3. Don't be afraid to search the internet for help
4. The answered notebook should be submitted in the following format: HW2_{id}.ipynb
5. **Before submission reset kernel and run all, make sure there are no running errors!!!**
6. When needed to give an answer, don't print it unless you were asked to, just write it at the end of the cell and you should see the output as it is.
7. Do not change variable names or tests. If you are asked to answer with a certain variable, use it and do not use a different name. Do not change the given tests if there are any. You can add new tests as you'd like, but leave the given test in the code as it is.

### Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from random import randint
from math import pi as PI
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

## Instructions
Complete code wherever there is a TODO. If you want to check yourself simply press ctrl+f and search for TODO 

# PART 1: Implementing linear regression
In order to better understand how linear regression works, and how the sklearn API works, in this part you shall partially implement the LinearRegression model.
Reminder: for linear regression, $\hat{f}(x)=w_0+w_1x_1+...+w_px_p$
See lecture to see how the $w_0,...,w_p$ are set (do not forget $w_0$)

a. Load the boston housing dataset we saw in recitation, as a DataFrame. Save the features in X_boston and the labels in y_boston, both should be either Series or DataFrame.

In [2]:
#insert code here
boston_data = pd.read_csv("boston_housing.csv")
feature_cols = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
X_boston = boston_data[feature_cols]
y_boston = boston_data["MEDV"]

b. Implement the fit & predict methods in MyLinearRegression model (without the sklearn package. using numpy and pandas only). Note that the original input X should not be changed outside the function so use the copy method before changing it. numpy has various mathematical functions which can assist you.

In [12]:
class MyLinearRegression(object):
    
    def __init__(self):
        self.coef_: np.array = None
        self.intercept_: float = None
    
    def fit(self, X, y):
        """
        Fits the 2D input X to the label y
        :param X: 2D input (either an np.array or a pd.DataFrame)
        :param y: 1D input (either np.array or pd.Series). Dimension must match first dimension of X
        :return: self. an instance of self
        """
        X_copy= X.copy()
        X_copy['for_inter']=1.
        w = np.matmul(np.linalg.pinv(X_copy), y)
        self.coef_ = w[:-1]
        self.intercept_ = w[-1]
        
    
    def predict(self, X):
        """
        predicts the label of each row in X
        :param X: 2D input (either an np.array or a pd.DataFrame)
        :return: 1D np.array with the predictions
        """
        return np.dot(X, self.coef_) + self.intercept_
        

c. Fit your model on the boston data. and use the predict method to make check the model predictions. <b>Print</b> out the models intercept, coefficients, and MSE. <b>DO NOT use sklearn to compute the MSE</b>.
<br>

In [13]:
# insert code here
reg= MyLinearRegression()
reg.fit(X_boston, y_boston)
predicts = reg.predict(X_boston)
print('Learned intercept = ', reg.intercept_)
print('Learned coefficients = ', reg.coef_)
print('Dataset MSE = ', np.mean(np.square(predicts-y_boston)))

Learned intercept =  36.45948838508984
Learned coefficients =  [-1.08011358e-01  4.64204584e-02  2.05586264e-02  2.68673382e+00
 -1.77666112e+01  3.80986521e+00  6.92224640e-04 -1.47556685e+00
  3.06049479e-01 -1.23345939e-02 -9.52747232e-01  9.31168327e-03
 -5.24758378e-01]
Dataset MSE =  21.894831181729206


d. Check if your answers asnwers are correct. This time, use sklearn.metrics and check your MSE. Print out the intercept, coefficients and MSE. Your answers from the previous section should be similar to your answers in this section.

In [15]:
# insert code here
sk_reg = LinearRegression()
sk_reg.fit(X_boston,y_boston)
print('Learned intercept = ', sk_reg.intercept_)
print('Learned coefficients = ', sk_reg.coef_)
print('Dataset MSE = ', mean_squared_error(sk_reg.predict(X_boston), y_boston))

Learned intercept =  36.45948838508985
Learned coefficients =  [-1.08011358e-01  4.64204584e-02  2.05586264e-02  2.68673382e+00
 -1.77666112e+01  3.80986521e+00  6.92224640e-04 -1.47556685e+00
  3.06049479e-01 -1.23345939e-02 -9.52747232e-01  9.31168327e-03
 -5.24758378e-01]
Dataset MSE =  21.894831181729202


e. Which features have positive impact on the label (increases it), and which have negative impact on it (decreasing it)? Print your answers. <br> do not measure the impact of the intercept in this section.

In [ ]:
print('Features with positive impact on the label: ', list(X_boston.columns[reg.coef_ > 0])) # TODO: Change 'None' the printed value to the correct value
print('Features with negative impact on the label: ', list(X_boston.columns[reg.coef_ < 0])) # TODO: Change 'None' the printed value to the correct value

f. Choose one feature, and explain intuitely (in markdown) why it makes sense it has a positive\negative impact on the label. For full description of the features, see recitation.

Click here on the cell and write your answer.

Answer: 

# PART 2: fitting polynomial linear regression
> In a real-life situation, it is generally not possible to explicitly compute the test MSE, bias, or variance for a statistical learning method. Nevertheless, one should always keep the bias-variance trade-off in mind.

    Page 36, An Introduction to Statistical Learning with Applications in R, 2014.
**Why is it that we cannot calculate the above propeties?** <br>
 Because we do not have the true mapping $ f(x) $ . all we observe is $ y $ </span>.
<br>In this section we shall illustrate the difference between train and test.
Recall in the lecture 3, We have introduced the polynomial linear regression, which is linear regression with polynomial features: $f_t(x)=w_0+w_1x+w_2x^2+...+w_tx^t$
<br>
The following PolynomialLinearRegression function acts as a constructor, and creates a polynomial model (with the fit & predict methods).

In [ ]:
def PolynomialLinearRegression(degree):
    return make_pipeline(PolynomialFeatures(degree),LinearRegression())

In this section, we create an experimental data.

In [ ]:
plt.style.use('ggplot')
function_title = 'y = cos(2πx) + Ɛ'
# for reproducability
np.random.seed(10)

# number of observations
NUM_OBS = 400

# predictors
x = np.linspace(0, 3, num = NUM_OBS)
# noise
eps = np.random.normal(0, 1, NUM_OBS)
# outcome
y = np.cos(2*PI*x) + eps

# plot
fig = plt.figure(figsize=(7,7))
ax = plt.axes()
ax.set_title(function_title)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(x, y, c = 'k')
plt.show()

#### a. Answer below: what is the true unknown function we used? what is the variance of the noise?

Click here on the cell and write your answer.

Answer:

To see the impact on the train and test set, we now split the data into train and test sets. we use 20% of the data for test.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)
# plot
fig = plt.figure(figsize = (7,7))
ax = plt.axes()
ax.set_title(function_title)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(x_train, y_train, label = "Train set")
ax.scatter(x_test, y_test, label = "Test set")
ax.legend()
plt.show()

#### b. Finding the best polynomial degree: 
The following function `plot polynomial degrees` tries out multiple degrees to fit the dataset. The function plots the points with their predictions on the train and test datasets, according to the different degrees, and the corresponding MSEs. <br>

You need to implement the function `choose_best_degree` which returns the best degree, given two lists containing mse results for the train and test sets. <br>

The function should return the degree which generalizes best, based on the given data we generated.

In [ ]:
def choose_best_degree(degrees_list,mse_train_lst, mse_test_lst) -> int:
    """
    Return the best degree of the model.
    What is the best degree? Hint: you only need to use one of the lists.
    
    mse_train_lst: list of mse for each degree of the train set
    mse_test_lst: list of mse for each degree of the test set
    degrees_list: list of degrees values
    """
    pass # TODO: replace this line with your code


In [ ]:
def plot_polynomial_degrees(degrees_list):
    degrees_list = sorted(degrees_list)
    train_mse = []
    test_mse = []
    x_train_2d = np.expand_dims(x_train, axis=1)
    x_test_2d = np.expand_dims(x_test, axis=1)
    train_spaces = np.expand_dims(np.linspace(x_train.min(), x_train.max(), 500), axis=1)
    test_spaces = np.expand_dims(np.linspace(x_test.min(), x_test.max(), 500), axis=1)
    colors = iter(plt.cm.rainbow(np.linspace(0, 1, len(degrees_list))))

    fig, axs = plt.subplots(2, 2, figsize = (20,20))

    axs[0, 0].set_title('Train Set Predictions')
    axs[1, 0].set_title('Test Set Predictions')
    axs[0, 1].set_title('Train Set MSE')
    axs[1, 1].set_title('Test Set MSE')

    axs[0, 0].set_xlabel("x train")
    axs[0, 0].set_ylabel("y")
    axs[0, 0].set_ylim([-3, 3])

    axs[1, 0].set_xlabel("x test")
    axs[1, 0].set_ylabel("y")
    axs[1, 0].set_ylim([-3, 3])

    axs[0, 1].set_xlabel("polynomial degree")
    axs[0, 1].set_ylabel("train MSE")

    axs[1, 1].set_xlabel("polynomial degree")
    axs[1, 1].set_ylabel("test MSE")

    axs[0,0].scatter(x_train, y_train, c = 'k', label = "y train")
    axs[1,0].scatter(x_test, y_test, c = 'k', label = "y test")

    for k in degrees_list:
        c = next(colors)

        # k-th degree polynomial coefficients
        reg = PolynomialLinearRegression(k)
        reg.fit(x_train_2d, y_train)

      # train and test k-th degree polynomial fit
        y_train_pred = reg.predict(x_train_2d)
        y_test_pred = reg.predict(x_test_2d)

        axs[0,0].plot(train_spaces[:,0],
                      reg.predict(train_spaces),
                      color = c, 
                      linewidth=3,
                      label = "deg: {}".format(k))

        axs[1,0].plot(train_spaces[:,0],
                      reg.predict(test_spaces),
                      color = c,
                      linewidth = 3,
                      label = "deg: {}".format(k))

      # train and test MSE of k-th degree polynomial fit 
        iter_train_mse = mean_squared_error(y_train_pred, y_train)
        iter_test_mse = mean_squared_error(y_test_pred, y_test)

        train_mse.append(iter_train_mse)
        test_mse.append(iter_test_mse)

      # plot train and test MSE of k-th degree polynomial fit
        axs[0,1].plot(k,
                    iter_train_mse,
                    color = c,
                    label = "deg: {}".format(k),
                    marker = 'D',
                    markersize = 12,
                    markeredgecolor = 'black',
                    markeredgewidth = 3)

        axs[1,1].plot(k,
                    iter_test_mse,
                    color = c,
                    label = "deg: {}".format(k),
                    marker = 'D',
                    markersize = 12,
                    markeredgecolor = 'black',
                    markeredgewidth = 3)

    # plot dashed line to interpolate MSE measures
    axs[0,1].plot(degrees_list, train_mse, 'k--')
    axs[1,1].plot(degrees_list, test_mse, 'k--')

    # draw legends
    axs[0,1].legend(loc = "upper right",
                    bbox_to_anchor = (1.2, 1.01),
                    prop = {'size': 12})
    axs[1,1].legend(loc = "upper right",
                    bbox_to_anchor = (1.2, 1.01),
                    prop = {'size': 12})
    
    chosen_degree = choose_best_degree(degrees_list,train_mse, test_mse)
    print("Chosen degree is: ", chosen_degree)

b. Below you can see an example usage plot_polynomial_degrees. 

In [ ]:
plot_polynomial_degrees(range(20))

c1. According to the results from section b, use plot_polynomial_degrees to plot only 3 models out of the options [0,...,20].
1. The model with the maximum bias
2. The model with the maximum variance
3. The model which generalizes best (the degree that was selected above)

Important: don't calculate maximum bias and variance.

In [ ]:
#TODO: write your code here
# for example:
# plot_polynomial_degrees([a,b,c]) 
# when a, b, c are the degrees you want to plot (numbers)

c2. Answer - Why did you choose the degree you chose for the model with the maximum bias? Why did you choose the degree you chose for the model with the maximum variance? Answer in the markdown cell below.

Click here on the cell and write your answer.

Answer: 